# TD 2 — Qualité et nettoyage des données (chapitres 3–4)

**Cours : Analyse et traitement des données** · Auteur : *Issa Gueye*

Jeu principal : `meteo_dakar.csv` (station météo simulée, 2020–2024) et `enquete_menages.csv`.

In [1]:
import io
from pathlib import Path

import numpy as np
import pandas as pd

DATA = next(p for p in [Path("data/brutes"), Path("../data/brutes"), Path("../../data/brutes")] if p.exists())

## Exercice 1 ⭐ — Lire un CSV « à la française »
Lisez correctement la chaîne ci-dessous : séparateur `;`, virgule décimale, `ND` et `-` = manquant, dates jour/mois/année.
Vérifiez que `pluie_mm` est numérique et `date` de type date.

In [2]:
brut = """date;station;pluie_mm;temp_c
01/08/2024;Dakar-Yoff;12,5;27,1
02/08/2024;Dakar-Yoff;ND;26,8
03/08/2024;Dakar-Yoff;0;-
04/08/2024;Dakar-Yoff;31,2;25,9
"""

In [3]:
df = pd.read_csv(io.StringIO(brut), sep=";", decimal=",", na_values=["ND", "-"], parse_dates=["date"], dayfirst=True)
print(df.dtypes); df

date        datetime64[us]
station                str
pluie_mm           float64
temp_c             float64
dtype: object


,date,station,pluie_mm,temp_c
0,2024-08-01,Dakar-Yoff,12.5,27.1
1,2024-08-02,Dakar-Yoff,NaN,26.8
2,2024-08-03,Dakar-Yoff,0.0,NaN
3,2024-08-04,Dakar-Yoff,31.2,25.9


## Exercice 2 ⭐⭐ — Audit de la météo
Chargez `meteo_dakar.csv`. Produisez :
1. le nombre et le pourcentage de manquants par colonne ;
2. le nombre de **jours absents** du fichier entre la première et la dernière date ;
3. le nombre de températures **physiquement impossibles** (< 0 °C ou > 50 °C à Dakar).

In [4]:
meteo = pd.read_csv(DATA / "meteo_dakar.csv", parse_dates=["date"])
print(pd.DataFrame({"n": meteo.isna().sum(), "%": (meteo.isna().mean() * 100).round(2)}))
attendu = pd.date_range(meteo["date"].min(), meteo["date"].max(), freq="D")
print("jours absents :", len(attendu.difference(meteo["date"])))
print("températures impossibles :", ((meteo["temp_moy_c"] < 0) | (meteo["temp_moy_c"] > 50)).sum())

               n     %
date           0  0.00
temp_moy_c    34  1.88
humidite_pct  34  1.88
precip_mm      0  0.00
vent_ms       20  1.11
jours absents : 20
températures impossibles : 4


## Exercice 3 ⭐⭐ — Nettoyer la météo
Écrivez une fonction `nettoyer_meteo(df)` qui :
1. remplace les températures impossibles par `NaN` ;
2. réindexe sur **tous** les jours (`asfreq("D")`) ;
3. interpole la température et l'humidité (méthode `"time"`, au plus 5 jours consécutifs) ;
4. ajoute une colonne booléenne `temp_imputee`.
Combien de valeurs de température restent manquantes après nettoyage ?

In [5]:
def nettoyer_meteo(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy().set_index("date").sort_index()
    d.loc[(d["temp_moy_c"] < 0) | (d["temp_moy_c"] > 50), "temp_moy_c"] = np.nan
    d = d.asfreq("D")
    d["temp_imputee"] = d["temp_moy_c"].isna()
    for col in ["temp_moy_c", "humidite_pct"]:
        d[col] = d[col].interpolate(method="time", limit=5)
    return d

propre = nettoyer_meteo(meteo)
print("restant manquant :", propre["temp_moy_c"].isna().sum(), "| imputées :", propre["temp_imputee"].sum())

restant manquant : 7 | imputées : 58


## Exercice 4 ⭐⭐ — Mécanisme de non-réponse
Dans `enquete_menages.csv`, le revenu manque pour certains ménages.
1. Calculez le taux de non-réponse par **région**, par **accès internet** et par **sexe du chef**.
2. La non-réponse vous semble-t-elle MCAR ? Argumentez en 3 lignes.

In [6]:
m = pd.read_csv(DATA / "enquete_menages.csv")
m["nr"] = m["revenu_mensuel_fcfa"].isna()
for col in ["region", "acces_internet", "sexe_chef"]:
    print(m.groupby(col)["nr"].mean().mul(100).round(1), "\n")
# Le taux varie fortement avec la région (Dakar, très urbaine, plus élevé) et l'accès internet (proxy du niveau d'éducation
# et du milieu urbain), mais pas avec le sexe du chef. La non-réponse dépend donc de variables observées :
# ce n'est pas MCAR ; l'hypothèse MAR (conditionnellement au milieu/éducation) est plausible, MNAR ne peut être exclu.

region
Dakar          16.9
Diourbel       10.3
Kaolack         7.5
Louga           8.8
Saint-Louis     9.1
Tambacounda    10.6
Thiès          10.6
Ziguinchor     10.8
Name: nr, dtype: float64 

acces_internet
Non     8.1
Oui    16.4
Name: nr, dtype: float64 

sexe_chef
F    11.6
H    11.9
Name: nr, dtype: float64 



## Exercice 5 ⭐⭐⭐ — Comparer des imputations
Sur les ménages dont le revenu est connu, masquez **artificiellement** 20 % des revenus **au hasard (MCAR)**,
puis comparez l'erreur absolue moyenne (sur les valeurs masquées) de :
(a) la médiane globale, (b) la médiane par `milieu × education_chef`, (c) `KNNImputer` sur log(revenu), log(dépenses), taille.
Recommencez avec un masquage **MAR** (35 % chez les urbains, 10 % chez les ruraux). Conclusion ?

In [7]:
from sklearn.impute import KNNImputer

def evaluer(masquer_p, graine=0):
    rng = np.random.default_rng(graine)
    c = m.dropna(subset=["revenu_mensuel_fcfa", "taille_menage"]).copy()
    c = c[c["taille_menage"] != 99]
    verite = c["revenu_mensuel_fcfa"].copy()
    masque = rng.random(len(c)) < masquer_p(c)
    c.loc[masque, "revenu_mensuel_fcfa"] = np.nan
    res = {}
    res["médiane globale"] = c["revenu_mensuel_fcfa"].fillna(c["revenu_mensuel_fcfa"].median())
    res["médiane par groupe"] = c["revenu_mensuel_fcfa"].fillna(
        c.groupby(["milieu", "education_chef"])["revenu_mensuel_fcfa"].transform("median"))
    X = np.column_stack([np.log(c["revenu_mensuel_fcfa"]), np.log(c["depenses_mensuelles_fcfa"]), c["taille_menage"]])
    res["KNN"] = pd.Series(np.exp(KNNImputer(n_neighbors=10).fit_transform(X)[:, 0]), index=c.index)
    return {k: np.abs(v[masque] - verite[masque]).mean().round(0) for k, v in res.items()}

print("MCAR :", evaluer(lambda c: 0.20))
print("MAR  :", evaluer(lambda c: np.where(c["milieu"] == "Urbain", 0.35, 0.10)))
# Les méthodes qui exploitent les autres variables (groupe, KNN avec les dépenses) sont nettement meilleures dans les deux
# cas ; KNN tire profit de la forte corrélation revenu–dépenses.

MCAR : {'médiane globale': np.float64(145274.0), 'médiane par groupe': np.float64(116551.0), 'KNN': np.float64(37157.0)}
MAR  : {'médiane globale': np.float64(173977.0), 'médiane par groupe': np.float64(139158.0), 'KNN': np.float64(49110.0)}


## Exercice 6 ⭐⭐ — Valeurs aberrantes
Sur la vitesse du vent (`vent_ms`) de la météo nettoyée :
1. comptez les valeurs signalées par la règle de Tukey (1,5 × IQR) et par le z-score robuste (|z| > 3,5) ;
2. affichez les 5 valeurs les plus élevées : erreurs ou phénomènes réels plausibles ? Que décidez-vous ?

In [8]:
v = propre["vent_ms"].dropna()
q1, q3 = v.quantile([.25, .75]); iqr = q3 - q1
tukey = (v < q1 - 1.5 * iqr) | (v > q3 + 1.5 * iqr)
med = v.median(); mad = (v - med).abs().median()
robuste = ((v - med).abs() / (1.4826 * mad)) > 3.5
print("Tukey :", tukey.sum(), "| MAD :", robuste.sum())
print(v.nlargest(5))
# Des vents de 10–11 m/s (≈ 40 km/h) sont tout à fait plausibles à Dakar en saison sèche (alizés, harmattan) :
# ce sont des valeurs réelles extrêmes, à conserver (pas de correction).

Tukey : 8 | MAD : 2
date
2024-01-24    10.5
2022-02-01    10.3
2020-06-24     9.4
2023-03-30     9.3
2024-09-30     9.2
Name: vent_ms, dtype: float64
